# Human Pose Estimation — OpenPose BODY_25 via `cv2.dnn`

Estimating human body pose with the BODY_25 model: heatmaps, Part Affinity
Fields, and both the bottom-up and the top-down pipeline.

All paths are relative to the project root, so Jupyter has to be started from
there. Kernel: **Python 3.11 (Computer Vision venv)**.

## What the network returns

BODY_25 is a fully convolutional Caffe graph. It takes an image and returns a
tensor of shape `(1, 78, H/8, W/8)`. The stride of 8 comes from four poolings in
the VGG-like backbone: every output cell covers an 8x8 square of the input.

Those 78 channels are three different things packed into one tensor:

| channels | what it is | how to read it |
|---|---|---|
| 0–24 | heatmaps of the 25 joints | value = probability that the joint is here |
| 25 | background | not needed, but it occupies a channel |
| 26–77 | PAF (Part Affinity Fields) | 26 limbs x 2 channels: the x and y component of a unit vector along the limb |

Heatmaps answer *where the joints are*, PAFs answer *which joints belong to the
same person*. The second question is why PAFs exist at all: with three people in
frame, the left-wrist heatmap has three peaks, and nothing else tells you whose
is whose.

Two approaches to keep apart:

- **bottom-up** — run the whole frame once, find every joint, group them with
  PAFs. Runtime does not depend on the number of people.
- **top-down** — run a person detector first, then pose on each crop. The crop is
  stretched to the network input, so small people "grow"; you pay per person.

---

## Block 1. Imports

In [ ]:
import os
import cv2
import numpy as np
import urllib.request          # joint_a bare `import urllib` does NOT give you this submodule
import matplotlib.pyplot as plt
import time

%matplotlib inline

---

## Block 2. Downloading the model, with a size check

Pulls the architecture (`pose_deploy.prototxt`, ~42 KB) and the weights
(`pose_iter_584000.caffemodel`, exactly 104,715,850 bytes) into
`models/pose_body25/`.

Why not a one-line `urlretrieve`:

1. The official CMU host (`posefs1.perception.cs.cmu.edu`) stopped resolving
   long ago, which kills every tutorial that uses it. The architecture comes
   from CMU's GitHub, the weights from a Hugging Face mirror.
2. 100 MB over a flaky link gets cut halfway. On a broken connection
   `urlretrieve` leaves a **truncated file that looks perfectly normal**:
   `os.path.isfile()` says `True`, the next run skips the download, and
   `readNetFromCaffe` then dies with an unreadable protobuf error.

So the function below resumes with a `Range` header and verifies the final size
against a known number instead of trusting that the file exists.

In [ ]:
MODEL_DIR = "models/pose_body25"
protoFile = os.path.join(MODEL_DIR, "pose_deploy.prototxt")
weightsFile = os.path.join(MODEL_DIR, "pose_iter_584000.caffemodel")

PROTO_URL = ("https://raw.githubusercontent.com/CMU-Perceptual-Computing-Lab/"
             "openpose/master/models/pose/body_25/pose_deploy.prototxt")
WEIGHTS_URL = "https://huggingface.co/dylanholmes/openpose-caffemodels/resolve/main/body25.caffemodel"
WEIGHTS_SIZE = 104_715_850     # anything smaller means the download was cut short


def download(source_url, destination_path, expected_size=None, attempts=10, timeout=60):
    """Download `source_url` into `destination_path`, resuming after a dropped connection."""
    os.makedirs(os.path.dirname(destination_path) or ".", exist_ok=True)

    for attempt in range(1, attempts + 1):
        bytes_downloaded = os.path.getsize(destination_path) if os.path.isfile(destination_path) else 0
        if expected_size and bytes_downloaded >= expected_size:
            return True
        if expected_size is None and bytes_downloaded:
            return True

        # Ask the server to continue from the byte the last attempt stopped at.
        headers = {"Range": f"bytes={bytes_downloaded}-"} if bytes_downloaded else {}
        try:
            with urllib.request.urlopen(urllib.request.Request(source_url, headers=headers), timeout=timeout) as response:
                # 206 = the server honoured Range, so appending is safe.
                # 200 = it ignored Range and resends from byte 0, so appending
                #       would corrupt the file - reset and overwrite instead.
                file_mode = "ab" if (bytes_downloaded and response.status == 206) else "wb"
                if file_mode == "wb":
                    bytes_downloaded = 0
                total_bytes = bytes_downloaded + int(response.headers.get("Content-Length", 0))
                last_reported_bytes = bytes_downloaded
                with open(destination_path, file_mode) as output_file:
                    while True:
                        chunk = response.read(1 << 20)
                        if not chunk:
                            break
                        output_file.write(chunk)
                        bytes_downloaded += len(chunk)
                        # Report every 10 MB, not every chunk: "\r" does not
                        # erase the line in Jupyter the way it does in joint_a
                        # terminal, so each print would become its own line.
                        if bytes_downloaded - last_reported_bytes >= 10 << 20:
                            last_reported_bytes = bytes_downloaded
                            print(f"  {os.path.basename(destination_path)}: {bytes_downloaded/1e6:6.1f} / {total_bytes/1e6:.1f} MB")
        except Exception as error:
            # A dropped connection is the normal case here, not joint_a failure.
            print(f"  attempt {attempt} interrupted: {error}")
            time.sleep(2)

    # Trust the size, not the mere existence of the file.
    return os.path.isfile(destination_path) and (not expected_size or os.path.getsize(destination_path) >= expected_size)


download(PROTO_URL, protoFile)
download(WEIGHTS_URL, weightsFile, WEIGHTS_SIZE)
print("prototxt:", os.path.getsize(protoFile), "bytes")
print("weights :", os.path.getsize(weightsFile), "bytes  (expected", WEIGHTS_SIZE, ")")

---

## Block 3. Loading the network

Two things worth knowing here. The argument order of `readNetFromCaffe` is
**prototxt first, weights second** (`readNetFromTensorflow` is the other way
round) — swap them and you get a protobuf error that looks exactly like the one
a truncated file gives.

And CPU is not a compromise here: there is no CUDA on a Mac, `cv2.dnn` does not
use Metal, and for this network the OpenCL target is usually slower than CPU.

In [ ]:
pose_net = cv2.dnn.readNetFromCaffe(protoFile, weightsFile)     # prototxt first, then weights
pose_net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
pose_net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

print("layers:", len(pose_net.getLayerNames()))                 # 261

---

## Block 4. BODY_25 constants

Three lists, and all three have to agree with each other.

**Joints** — the index is the heatmap channel:

```
0 Nose      5 LShoulder  10 RKnee    15 REye   20 LSmallToe
1 Neck      6 LElbow     11 RAnkle   16 LEye   21 LHeel
2 RShoulder 7 LWrist     12 LHip     17 REar   22 RBigToe
3 RElbow    8 MidHip     13 LKnee    18 LEar   23 RSmallToe
4 RWrist    9 RHip       14 LAnkle   19 LBigToe 24 RHeel
```

R/L are from the point of view of **the person in the frame**, not the viewer.

`POSE_PAIRS` holds the 26 limbs the network computes PAFs for, and `MAP_IDX`
holds the two PAF channels of each limb relative to `PAF_OFFSET`. Both come from
`poseParameters.cpp` in OpenPose and their order is locked together:
`MAP_IDX[k]` are the channels for `POSE_PAIRS[k]`. Reorder either one and the
skeleton will connect a knee to an ear — a silent failure, with no exception.

In [ ]:
# Index = heatmap channel. R/L are from the point of view of the person in frame.
BODY_PARTS = ["Nose", "Neck", "RShoulder", "RElbow", "RWrist", "LShoulder",
              "LElbow", "LWrist", "MidHip", "RHip", "RKnee", "RAnkle", "LHip",
              "LKnee", "LAnkle", "REye", "LEye", "REar", "LEar", "LBigToe",
              "LSmallToe", "LHeel", "RBigToe", "RSmallToe", "RHeel"]

N_POINTS = len(BODY_PARTS)     # 25
PAF_OFFSET = N_POINTS + 1      # 0-24 are joints, 25 is background, 26+ are PAFs

In [ ]:
# The order of these two lists is locked together: MAP_IDX[limb_index] are the PAF
# channels of POSE_PAIRS[limb_index]. Reorder one of them and the skeleton connects joint_a
# knee to an ear - silently, without raising anything.
POSE_PAIRS = [[1, 8], [1, 2], [1, 5], [2, 3], [3, 4], [5, 6], [6, 7], [8, 9],
              [9, 10], [10, 11], [8, 12], [12, 13], [13, 14], [1, 0], [0, 15],
              [15, 17], [0, 16], [16, 18], [2, 17], [5, 18], [14, 19], [19, 20],
              [14, 21], [11, 22], [22, 23], [11, 24]]

MAP_IDX = [[0, 1], [14, 15], [22, 23], [16, 17], [18, 19], [24, 25], [26, 27],
           [6, 7], [2, 3], [4, 5], [8, 9], [10, 11], [12, 13], [30, 31],
           [32, 33], [36, 37], [34, 35], [38, 39], [20, 21], [28, 29], [40, 41],
           [42, 43], [44, 45], [46, 47], [48, 49], [50, 51]]

# Pairs [2,17] and [5,18] (shoulder-to-ear): useful for grouping people, but on
# the drawing they strike joint_a line straight across the face.
SKIP_RENDER = {18, 19}

# One BGR colour per limb, spread evenly around the hue circle.
COLORS = [tuple(int(channel_value) for channel_value in cv2.cvtColor(
              np.uint8([[[limb_index * 180 // len(POSE_PAIRS), 255, 255]]]), cv2.COLOR_HSV2BGR)[0, 0])
          for limb_index in range(len(POSE_PAIRS))]

**Structural check.** The anatomy check comes in block 7, once `draw_pose`
exists — but this one is free and catches half the mistakes right here. The
second assert is the useful one: a duplicated or missing PAF channel fails now
instead of showing up as a crooked skeleton three blocks later.

In [ ]:
assert len(POSE_PAIRS) == len(MAP_IDX) == 26
assert sorted(channel for limb_pair in MAP_IDX for channel in limb_pair) == list(range(52))   # every PAF channel exactly once
assert max(joint_index for limb_pair in POSE_PAIRS for joint_index in limb_pair) == N_POINTS - 1      # no joint index above 24
assert len(COLORS) == len(POSE_PAIRS)
print("constants OK")

A small helper used everywhere below — matplotlib expects RGB, OpenCV gives BGR.

In [ ]:
def show(image, title="", figsize=(11, 8)):
    plt.figure(figsize=figsize)
    plt.imshow(image[:, :, ::-1])      # BGR -> RGB for matplotlib
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

---

## Block 5. One forward pass

Every argument of `blobFromImage` has a reason:

- `1/255.0` — the model was trained on inputs in the range [0, 1];
- `(in_width, in_height)` — the width follows the frame's aspect ratio and is
  rounded to a multiple of 8. Break the ratio and the person is squeezed
  horizontally, which costs accuracy; skip the rounding and OpenCV pads the size
  itself, shifting the maps by half a cell;
- `(0, 0, 0)` — no mean subtraction, the model does not expect it;
- **`swapRB=False`** — the Caffe model was trained on BGR and `cv2.imread`
  already returns BGR. Set it to `True` (as TensorFlow models want) and the
  network still finds *something*, just worse and less reliably. A classic
  silent bug;
- `crop=False` — otherwise OpenCV centre-crops the frame instead of resizing it.

The last line stretches all 78 maps back to frame size, so peak coordinates are
already in original pixels and nothing below has to multiply by `w/out_w`. The
price is memory: 78 float maps at full frame size.

In [ ]:
def run_pose(pose_net, image, input_height=368):
    image_height, image_width = image.shape[:2]
    # Width follows the frame aspect ratio, rounded to joint_a multiple of the
    # network stride (8).
    input_width = max(8, int(round(input_height * image_width / image_height / 8)) * 8)

    input_blob = cv2.dnn.blobFromImage(image, 1 / 255.0, (input_width, input_height),
                                 (0, 0, 0), swapRB=False, crop=False)

    pose_net.setInput(input_blob)
    raw_output = pose_net.forward()             # (1, 78, input_height/8, input_width/8)

    # Stretch every map back to frame size, so peaks are already in the pixel
    # coordinates of the original image.
    return np.stack([cv2.resize(raw_output[0, channel_index], (image_width, image_height)) for channel_index in range(raw_output.shape[1])])

**Visual check.** Expect `~0.7–1.2 s` and `pose_maps.shape == (78, 342, 548)`.

Nothing is drawn on the frame yet — `run_pose` does not modify `image`, and
`draw_pose` only appears in block 7. What this cell proves is that the maps come
back in frame pixels and that the image itself loaded correctly.

In [ ]:
image = cv2.imread("data/images/messi5.jpg")

started_at = time.time()
pose_maps = run_pose(pose_net, image)
print(f"{time.time() - started_at:.2f} s   im.shape={image.shape}   maps.shape={pose_maps.shape}")

# The real check here. Without the cv2.resize inside run_pose the shape would be
# (78, 46, 69), and every coordinate computed below would be off by joint_a factor of 8.
assert pose_maps.shape == (78,) + image.shape[:2]

# Plain input frame - no lines yet. This only confirms the image decoded and the
# BGR->RGB swap in show() is right (the grass has to look green, not red).
show(image, "input frame")

In [ ]:
def show_map (image, heat_map, title="", alpha=0.6):
    plt.figure(figsize=(11,8))
    plt.imshow(image[:,:,::-1])
    plt.imshow(heat_map,alpha=alpha,cmap="jet")
    plt.axis("off")
    plt.title(f"{title} max={heat_map.max():.2f}")
    plt.show()


In [ ]:
pose_maps = run_pose(pose_net, image)

for part_name in ["Nose","RWrist","LAnkle"]:
    part_index = BODY_PARTS.index(part_name)
    show_map(image,pose_maps[part_index], f"heatmap {part_name}")

In [ ]:
limb_index = POSE_PAIRS.index([3,4])
paf_magnitude = np.hypot(pose_maps[PAF_OFFSET + MAP_IDX[limb_index][0]],
              pose_maps[PAF_OFFSET + MAP_IDX[limb_index][1]])
show_map(image,paf_magnitude,"PAF RElbow->RWrist")

In [ ]:
for limb_pair in ([3,4],[10,11],[1,8]):
    limb_index = POSE_PAIRS.index(limb_pair)
    paf_magnitude = np.hypot(pose_maps[PAF_OFFSET + MAP_IDX[limb_index][0]],
                  pose_maps[PAF_OFFSET + MAP_IDX[limb_index][1]])
    show_map(image,paf_magnitude,f"PAF {BODY_PARTS[limb_pair[0]]} -> {BODY_PARTS[limb_pair[1]]}")

In [ ]:
def keypoints_single(pose_maps, threshold=0.1):
    points = []
    for joint_index in range(N_POINTS):
        _, confidence, _, location = cv2.minMaxLoc(pose_maps[joint_index])
        points.append((location[0], location[1], confidence) if confidence > threshold else None)
    return points

In [ ]:
points = keypoints_single(run_pose(pose_net,image))
missing_parts = [BODY_PARTS[joint_index] for joint_index, point in enumerate(points) if point is None]
print(f"found {N_POINTS - len(missing_parts)}/{N_POINTS}, lost: {missing_parts}")

In [ ]:
def get_keypoints(prob_map, threshold=0.1):
    smoothed_map = cv2.GaussianBlur(prob_map, (3,3),0,0)
    binary_mask = np.uint8(smoothed_map > threshold)

    keypoints = []
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours:
        blob_mask = cv2.fillConvexPoly(np.zeros(binary_mask.shape), contour, 1)
        _,_,_, location = cv2.minMaxLoc(smoothed_map * blob_mask)
        keypoints.append(location + (prob_map[location[1], location[0]],))
    return keypoints

In [ ]:
def detect_keypoints(pose_maps, threshold=0.1):
    candidates_by_joint, keypoints_list, next_id = [], [], 0
    for joint_index in range(N_POINTS):
        candidates_with_id = []
        for keypoint in get_keypoints(pose_maps[joint_index],threshold):
            keypoints_list.append(keypoint[:3])
            candidates_with_id.append(keypoint+(next_id,))
            next_id += 1
        candidates_by_joint.append(candidates_with_id)
    return candidates_by_joint, np.array(keypoints_list).reshape(-1,3)

In [ ]:
soldiers_image = cv2.imread("data/images/solders.jpg")
pose_maps = run_pose(pose_net, image)
candidates_by_joint, keypoints_list = detect_keypoints(pose_maps)

print("Candidates amount:", len(keypoints_list))
for joint_index, joint_candidates in enumerate(candidates_by_joint):
    if len(joint_candidates) > 1:
        print(f" {BODY_PARTS[joint_index]:10} x {len(joint_candidates)}")

annotated_image = image.copy()
for joint_candidates in candidates_by_joint: 
    for x_coordinate, y_coordinate, confidence, keypoint_id in joint_candidates:
        cv2.circle(annotated_image,(x_coordinate,y_coordinate), 4, (0,255,255), -1)
show(annotated_image, f"{len(keypoints_list)} canditates")

In [ ]:
def get_valid_pairs(pose_maps, candidates_by_joint, sample_count=10, paf_threshold=0.1, confidence_threshold=0.7):
    """Greedily match joint candidates into limbs, one limb type at a time.

    Returns a list of length len(POSE_PAIRS); element k is an (N, 3) array of
    [keypoint_id_a, keypoint_id_b, score] rows - the connections accepted for POSE_PAIRS[limb_index].
    The ids are the ones detect_keypoints handed out, not coordinates.
    """
    valid_pairs = []

    for limb_index in range(len(POSE_PAIRS)):
        paf_x_channel = pose_maps[PAF_OFFSET + MAP_IDX[limb_index][0]]
        paf_y_channel = pose_maps[PAF_OFFSET + MAP_IDX[limb_index][1]]

        candidates_a = candidates_by_joint[POSE_PAIRS[limb_index][0]]
        candidates_b = candidates_by_joint[POSE_PAIRS[limb_index][1]]

        # Nothing on one end - no limbs of this type. The empty entry still has
        # to be appended, or index limb_index stops lining up with POSE_PAIRS.
        if not candidates_a or not candidates_b:
            valid_pairs.append(np.empty((0, 3)))
            continue

        accepted_pairs, used_b_indices = [], set()

        for point_a in candidates_a:
            # Reset per candidate A, otherwise the first match blocks the rest.
            best_score, best_candidate_b = -1.0, -1

            for candidate_b_index, point_b in enumerate(candidates_b):
                if candidate_b_index in used_b_indices:            # one B belongs to one person only
                    continue

                direction = np.subtract(point_b[:2], point_a[:2])
                distance = np.linalg.norm(direction)
                if distance < 1e-6:            # two detections on the same pixel
                    continue
                direction = direction / distance

                # Sample the PAF along the straight line from A to B.
                sample_x_coordinates = np.linspace(point_a[0], point_b[0], sample_count)
                sample_y_coordinates = np.linspace(point_a[1], point_b[1], sample_count)
                vectors = np.array([[paf_x_channel[int(round(sample_y)), int(round(sample_x))],
                                     paf_y_channel[int(round(sample_y)), int(round(sample_x))]]
                                    for sample_x, sample_y in zip(sample_x_coordinates, sample_y_coordinates)])

                # Projection of each sampled PAF vector onto the A->B direction.
                scores = vectors @ direction
                mean_score = scores.mean()

                # Two different gates: paf_threshold filters joint_a single sample,
                # confidence_threshold demands that most samples pass it.
                if (np.sum(scores > paf_threshold) / sample_count) > confidence_threshold \
                        and mean_score > best_score:
                    best_score, best_candidate_b = mean_score, candidate_b_index

            if best_candidate_b != -1:
                used_b_indices.add(best_candidate_b)
                accepted_pairs.append([point_a[3], candidates_b[best_candidate_b][3], best_score])

        valid_pairs.append(np.array(accepted_pairs).reshape(-1, 3))

    return valid_pairs

In [ ]:
valid_pairs = get_valid_pairs(pose_maps, candidates_by_joint)

assert len(valid_pairs) == len(POSE_PAIRS)      # limb_index must still index POSE_PAIRS
print(f"accepted limbs: {sum(len(pairs_for_limb) for pairs_for_limb in valid_pairs)} of {len(keypoints_list)} candidates\n")

for limb_index, limb_pairs in enumerate(valid_pairs):
    if len(limb_pairs):
        joint_a, joint_b = POSE_PAIRS[limb_index]
        print(f"  {BODY_PARTS[joint_a]:>10} -> {BODY_PARTS[joint_b]:<11} {len(limb_pairs)}  "
              f"scores={np.round(limb_pairs[:, 2], 2).tolist()}")

In [ ]:
valid_pairs = get_valid_pairs(pose_maps, candidates_by_joint)
print("Valid pairs", sum(len(pairs_for_limb) for pairs_for_limb in valid_pairs))

annotated_image = image.copy()
for limb_index , limb_pairs in enumerate(valid_pairs):
    if limb_index in SKIP_RENDER:
        continue
    for keypoint_id_a, keypoint_id_b, score in limb_pairs:
        start_x, start_y = keypoints_list[int(keypoint_id_a),:2]
        end_x, end_y = keypoints_list[int(keypoint_id_b), :2]
        cv2.line(annotated_image, (int(start_x), int(start_y)), (int(end_x), int(end_y)), COLORS[limb_index],2)
show(annotated_image, "valid pairs before grouping")

In [ ]:
def group_people(valid_pairs, keypoints_list):
    people = np.empty((0, N_POINTS + 1)) 

    for limb_index, (joint_a,joint_b) in enumerate(POSE_PAIRS):
        for keypoint_id_a, keypoint_id_b, score in valid_pairs[limb_index]:
            existing_person_index = next((person_index for person_index, person in enumerate(people) if person[joint_a] == keypoint_id_a), -1)
            if existing_person_index != -1 :
                people[existing_person_index][joint_b] = keypoint_id_b
                people[existing_person_index][-1] += keypoints_list[int(keypoint_id_b), 2] + score 
            elif limb_index < 17:
                person_row = np.full(N_POINTS +1, -1.0)
                person_row[joint_a] , person_row[joint_b] = keypoint_id_a, keypoint_id_b
                person_row[-1] = keypoints_list[[int(keypoint_id_a),int(keypoint_id_b)], 2].sum() + score
                people = np.vstack([people, person_row])

    return people 

In [ ]:
def draw_people(image, people, keypoints_list, thickness=2, radius=4):
    """Draw one skeleton per row of `people` on a copy of `im`.

    Colour comes from COLORS[limb_index], the limb index - so the same body part is the
    same colour on every person. That is what makes a broken POSE_PAIRS/MAP_IDX
    pairing visible: a forearm in the colour of a shin means the lists slipped.
    """
    annotated_image = image.copy()

    for person in people:
        for limb_index, (joint_a, joint_b) in enumerate(POSE_PAIRS):
            if limb_index in SKIP_RENDER:              # shoulder-to-ear: joint_a line across the face
                continue

            # -1 means this person has no candidate for that joint.
            keypoint_id_a, keypoint_id_b = int(person[joint_a]), int(person[joint_b])
            if keypoint_id_a < 0 or keypoint_id_b < 0:
                continue

            start_x, start_y = keypoints_list[keypoint_id_a, :2].astype(int)
            end_x, end_y = keypoints_list[keypoint_id_b, :2].astype(int)
            cv2.line(annotated_image, (start_x, start_y), (end_x, end_y), COLORS[limb_index], thickness, cv2.LINE_AA)

        # Joints last, so the dots sit on top of the lines, not under them.
        for joint_index in range(N_POINTS):
            keypoint_id = int(person[joint_index])
            if keypoint_id >= 0:
                joint_x, joint_y = keypoints_list[keypoint_id, :2].astype(int)
                cv2.circle(annotated_image, (joint_x, joint_y), radius, (255, 255, 255), -1, cv2.LINE_AA)

    return annotated_image

In [ ]:
started_at = time.time()
pose_maps = run_pose(pose_net, soldiers_image)
candidates_by_joint, keypoints_list = detect_keypoints(pose_maps)
valid_pairs = get_valid_pairs(pose_maps, candidates_by_joint)
people = group_people(valid_pairs, keypoints_list)

MIN_JOINTS = 8
joint_counts = (people[:, :-1] >= 0).sum(axis=1)
strong_people = people[joint_counts >= MIN_JOINTS]

print(f"{time.time() - started_at:.2f}s  candidates: {len(keypoints_list)}  "
      f"rows: {len(people)} -> {len(strong_people)}")

show(draw_people(soldiers_image, strong_people, keypoints_list), "bottom-up")

In [ ]:
basketball_image = cv2.imread("data/images/basketball1.png")

for input_height_value in (256, 368, 512):
    started_at = time.time()
    pose_maps = run_pose(pose_net, basketball_image, input_height = input_height_value)
    candidates_by_joint, keypoints_list = detect_keypoints(pose_maps)
    valid_pairs = get_valid_pairs(pose_maps, candidates_by_joint)
    people = group_people(valid_pairs, keypoints_list)
    show(draw_people(basketball_image, people, keypoints_list),
         f"in_height={input_height_value} {time.time()-started_at:.2f} with skelets: {len(people)}"
        )

In [ ]:
def pose_top_down(image, input_height=368, detection_threshold=0.4, padding_ratio=0.15, draw_boxes=True):
    annotated_image = image.copy()
    for box_x1, box_y1, box_x2, box_y2, score in detect_persons(image, detection_threshold):
        margin = int(padding_ratio * max(box_x2 - box_x1, box_y2 - box_y1))
        crop_x1, crop_y1 = max(0, box_x1-margin),max(0, box_y1-margin)
        crop_x2, crop_y2 = min(image.shape[1], box_x2+margin), min(image.shape[0], box_y2 + margin)
        person_crop = image[crop_y1:crop_y2,crop_x1:crop_x2]
        if person_crop.size == 0:
            continue

        points = keypoints_single(run_pose(pose_net, person_crop, input_height))
        points = [None if point is None else (point[0] + crop_x1, point[1] + crop_y1, point[2]) for point in points]

        line_scale = max(1,(box_y2-box_y1)//60)
        if draw_boxes:
            cv2.rectangle(annotated_image, (box_x1, box_y1), (box_x2, box_y2), (0,255,255), 1)
        annotated_image = draw_pose(annotated_image, points, radius=line_scale + 1, thickness=line_scale)
        return annotated_image

In [ ]:
capture = cv2.VideoCapture("videos/vtest.mp4")
capture.set(cv2.CAP_PROP_POS_FRAMES, 300)
has_frame, frame = capture.read()
capture.release()
assert has_frame, "frame wasn't captured"

started_at = time.time()
pose_maps = run_pose(pose_net, frame)
candidates_by_joint, keypoints_list = detect_keypoints(pose_maps)
valid_pairs = get_valid_pairs(pose_maps, candidates_by_joint)
people = group_people(valid_pairs, keypoints_list)
show(draw_people(frame, people, keypoints_list), f"bottom-up {time.time() - started_at:.2f} s")

In [ ]:
def draw_pose(image, points, radius=4, thickness=2):
    """Draw one skeleton from a keypoints_single() list onto a copy of `im`.

    `points[joint_index]` is either (x, y, conf) or None - the top-down path needs the
    None case constantly, because a crop often cuts off feet or an ear.
    """
    annotated_image = image.copy()

    for limb_index, (joint_a, joint_b) in enumerate(POSE_PAIRS):
        if limb_index in SKIP_RENDER:
            continue
        if points[joint_a] is None or points[joint_b] is None:
            continue
        point_a = (int(points[joint_a][0]), int(points[joint_a][1]))
        point_b = (int(points[joint_b][0]), int(points[joint_b][1]))
        cv2.line(annotated_image, point_a, point_b, COLORS[limb_index], thickness, cv2.LINE_AA)

    for point in points:
        if point is not None:
            cv2.circle(annotated_image, (int(point[0]), int(point[1])), radius, (255, 255, 255), -1, cv2.LINE_AA)

    return annotated_image

In [ ]:
DET_DIR = "models/ssd_mobilenet_v2_coco_2018_03_29"

# A SECOND network. Do not name it `net` - that one is BODY_25, and run_pose
# takes it as an argument. Overwrite `net` and every pose cell above breaks.
person_detector = cv2.dnn.readNetFromTensorflow(
    os.path.join(DET_DIR, "frozen_inference_graph.pb"),
    os.path.join(DET_DIR, "ssd_mobilenet_v2_coco_2018_03_29.pbtxt"))


def detect_persons(image, threshold=0.4):
    """Return [(x1, y1, x2, y2, score), ...] for people only."""
    image_height, image_width = image.shape[:2]

    # swapRB=True here, unlike run_pose: this one is joint_a TensorFlow model trained
    # on RGB, while BODY_25 is Caffe and wants BGR.
    input_blob = cv2.dnn.blobFromImage(image, 1.0, (300, 300), (0, 0, 0), swapRB=True, crop=False)
    person_detector.setInput(input_blob)
    detections = person_detector.forward()          # (1, 1, N, 7)

    person_boxes = []
    for detection_index in range(detections.shape[2]):
        if int(detections[0, 0, detection_index, 1]) != 1:        # COCO class 1 = person
            continue
        score = float(detections[0, 0, detection_index, 2])
        if score < threshold:
            continue
        # Columns 3..6 are normalised to [0, 1] - multiply by frame size.
        box_x1, box_y1 = int(detections[0, 0, detection_index, 3] * image_width), int(detections[0, 0, detection_index, 4] * image_height)
        box_x2, box_y2 = int(detections[0, 0, detection_index, 5] * image_width), int(detections[0, 0, detection_index, 6] * image_height)
        person_boxes.append((max(0, box_x1), max(0, box_y1), min(image_width, box_x2), min(image_height, box_y2), score))

    return person_boxes

In [ ]:
source_path, output_path = "outputs/pose-vtest.mp4", "outputs/pose-vtest-h264.mp4"

capture = cv2.VideoCapture(source_path)
frames_per_second = capture.get(cv2.CAP_PROP_FPS)
frame_size = (int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)))

# avc1 = H.264. This is the only reason the old file would not play.
video_writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"avc1"), frames_per_second, frame_size)
assert video_writer.isOpened(), "no H.264 encoder in this OpenCV build"

while True:
    has_frame, frame = capture.read()
    if not has_frame:
        break
    video_writer.write(frame)

capture.release()
video_writer.release()
print(f"{os.path.getsize(output_path) / 1e6:.2f} MB")

Video(output_path, embed=True, width=720)

In [ ]:
capture = cv2.VideoCapture(0)
assert capture.isOpened(), "camera did not open"

# The first frames come out almost black: the Mac camera needs joint_a moment to
# settle its auto-exposure. Read joint_a few and throw them away.
for _ in range(10):
    capture.read()

has_frame, frame = capture.read()
capture.release()
assert has_frame, "camera opened but gave no frame"
print("Camera:", has_frame, frame.shape)

frame = cv2.flip(frame, 1)
show(draw_pose(frame, keypoints_single(run_pose(pose_net, frame, input_height=192))),
     "one frame from camera")